# TTBB Training on Colab — Woodruff North

Trains a PPO agent on the SoccerTwos `team_vs_policy` task with the CEIA baseline as a frozen opponent.

**Runtime:** A100 or H100 (Runtime → Change runtime type → GPU → A100)

**Run order:** cells top to bottom. `condacolab.install()` triggers a kernel restart; re-run from the *next* cell after the restart banner appears.

**Before starting:** replace `GITHUB_REPO_URL` in cell 5 with your pushed repo URL.

## 1. GPU check

In [ ]:
!nvidia-smi -L
!nvidia-smi --query-gpu=name,memory.total --format=csv

## 2. Install conda (triggers kernel restart)

After the restart banner, **skip this cell and continue from cell 3**.

In [ ]:
!pip install -q condacolab
import condacolab
condacolab.install()

## 3. System packages for headless Unity

In [ ]:
import subprocess, shutil, os
if shutil.which('Xvfb') is None:
    !apt-get install -y --no-install-recommends xvfb > /dev/null 2>&1
if shutil.which('Xvfb') is None:
    raise SystemExit('Xvfb still not found')
subprocess.Popen(['Xvfb', ':99', '-screen', '0', '1280x720x24'])
os.environ['DISPLAY'] = ':99'
print('xvfb ready on :99:', shutil.which('Xvfb'))

## 4. Create Python 3.8 env + install pinned deps

In [ ]:
!conda create -n ttbb python=3.8 -y -q 2>&1 | tail -5
%env PATH=/usr/local/envs/ttbb/bin:/usr/local/bin:/usr/bin:/bin
!python --version

In [ ]:
# pinned pip first per starter README
!pip install --quiet pip==23.3.2 setuptools==65.5.0 wheel==0.38.4
!pip cache purge > /dev/null

In [ ]:
# Core training stack — versions match the PACE conda env exactly
!pip install --quiet --extra-index-url https://download.pytorch.org/whl/cu113 \
    torch==1.11.0+cu113 torchvision==0.12.0+cu113
!pip install --quiet \
    soccer-twos==0.1.14 \
    ray==1.4.0 \
    mlagents-envs==0.27.0 \
    numpy==1.19.5 \
    protobuf==3.20.3 \
    pydantic==1.10.13 \
    prometheus-client==0.11.0
# Sanity import
!python -c 'import soccer_twos, ray, torch; print("torch", torch.__version__, "cuda", torch.cuda.is_available(), "ray", ray.__version__, "soccer_twos OK")'

## 5. Clone the repo

Replace `GITHUB_REPO_URL` below with the URL you pushed to.

In [ ]:
GITHUB_REPO_URL = 'https://github.com/Luca65537/soccer-twos-woodruffnorth.git'
!git clone $GITHUB_REPO_URL /content/repo
%cd /content/repo/training
!ls -la

## 6. Mount Google Drive for checkpoint persistence

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
!mkdir -p /content/drive/MyDrive/soccer_twos/ray_results
# Symlink so ray_results land on Drive
!ln -sfn /content/drive/MyDrive/soccer_twos/ray_results /content/repo/training/ray_results
!ls -la /content/repo/training/ray_results

## 7. Smoke test — one match vs baseline

This confirms: Unity binary runs under Xvfb, the frozen-baseline env wrapper steps without errors, and rewards flow through.

In [ ]:
!xvfb-run -a -s '-screen 0 1280x720x24' python -u smoke_ttbb.py

## 8. Launch training

20M env steps, or 8h wall-clock (whichever is first). On an A100 expect ~6 h total. Checkpoints save to Drive every 10 iterations.

In [ ]:
# bump batch size slightly for A100/H100 throughput
!sed -i 's/"num_workers": 2/"num_workers": 4/' train_vs_baseline.py
!sed -i 's/"num_envs_per_worker": NUM_ENVS_PER_WORKER,$/"num_envs_per_worker": NUM_ENVS_PER_WORKER,/' train_vs_baseline.py
!sed -i 's/NUM_ENVS_PER_WORKER = 2/NUM_ENVS_PER_WORKER = 3/' train_vs_baseline.py
!grep -E 'num_workers|NUM_ENVS_PER_WORKER|timesteps_total|time_total_s' train_vs_baseline.py

In [ ]:
!xvfb-run -a -s '-screen 0 1280x720x24' python -u train_vs_baseline.py 2>&1 | tee /content/drive/MyDrive/soccer_twos/train_log_$(date +%Y%m%d_%H%M).txt

## 9. After training — extract best checkpoint to standalone format

Saves a `checkpoint.pth` matching `agent_vanilla/model.py` so we can drop it into a fresh `WOODRUFF_NORTH_AGENT/` submission folder.

In [ ]:
import glob, os
ckpts = sorted(glob.glob('ray_results/ttbb_long/*/checkpoint_*/checkpoint-*'))
ckpts = [c for c in ckpts if not c.endswith('.tune_metadata')]
assert ckpts, 'No checkpoints found'
latest = ckpts[-1]
print('Extracting:', latest)
!python extract_ttbb_to_standalone.py '$latest' /content/drive/MyDrive/soccer_twos/ttbb_checkpoint.pth